# Moving-ball rollout experiment

In [1]:
import numpy as np

def frame(x, y, side):
    # Keep identical motion in the central 128 x 128 crop.
    pad = (side - 128) // 2
    rows, cols = np.ogrid[:side, :side]
    return (cols - x - pad)**2 + (rows - y - pad)**2 <= 3**2

def center(mask):
    rows, cols = np.nonzero(mask)
    return np.array([cols.mean(), rows.mean()])

rng = np.random.default_rng(42)
trials = [(int(rng.integers(36, 92)), int(rng.integers(10, 118)),
           int(rng.choice([-1, 1]))) for _ in range(32)]
records = []
for x, y, sign in trials:
    for side in [128, 256]:
        last = frame(x, y, side)
        velocity = center(last) - center(frame(x - 2 * sign, y, side))
        for h in [1, 4, 8, 16]:
            truth = frame(x + 2 * sign * h, y, side)
            coast = np.roll(last, (velocity * h).astype(int)[::-1], axis=(0, 1))
            for name, pred in [('freeze', last), ('velocity', coast)]:
                mismatch = 100 * np.mean(pred != truth)
                distance = np.linalg.norm(center(pred) - center(truth))
                records.append((side, h, name, mismatch, distance))

print('side  step  predictor  mismatch(%)  center_error(px)')
for side in [128, 256]:
    for h in [1, 4, 8, 16]:
        for name in ['freeze', 'velocity']:
            pairs = [row[3:] for row in records if row[:3] == (side, h, name)]
            mismatch, distance = np.mean(pairs, axis=0)
            print(f'{side:4}  {h:4}  {name:8}  {mismatch:11.4f}  {distance:16.1f}')


side  step  predictor  mismatch(%)  center_error(px)
 128     1  freeze         0.1465               2.0
 128     1  velocity       0.0000               0.0
 128     4  freeze         0.3540               8.0
 128     4  velocity       0.0000               0.0
 128     8  freeze         0.3540              16.0
 128     8  velocity       0.0000               0.0
 128    16  freeze         0.3540              32.0
 128    16  velocity       0.0000               0.0
 256     1  freeze         0.0366               2.0
 256     1  velocity       0.0000               0.0
 256     4  freeze         0.0885               8.0
 256     4  velocity       0.0000               0.0
 256     8  freeze         0.0885              16.0
 256     8  velocity       0.0000               0.0
 256    16  freeze         0.0885              32.0
 256    16  velocity       0.0000               0.0
